In [97]:
import pandas as pd
import numpy as np

In [98]:
import langchain

In [99]:
!pip install langchain-community pypdf
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [100]:
### We would load the Adobe brand guidelines PDF###
loader=PyPDFLoader("/content/Adobe_Brand_Voice.pdf")
documents=loader.load()

In [101]:
def clean_text(text):
    # Basic cleaning: remove extra spaces, line breaks
    text = text.replace("\n", " ").strip()
    return " ".join(text.split())

cleaned_texts = [clean_text(doc.page_content) for doc in documents]

In [102]:
cleaned_texts

["ADOBE Brand Voice & Tone Guidelines — For RAG Integration MISSION Changing the world through personalized digital experiences. Adobe empowers everyone — from individual creators to global enterprises — to bring their ideas to life through creativity and technology. Nurturing creativity is at the heart of everything Adobe does, for employees as well as the individuals, businesses, and communities it serves. BRAND BELIEF No matter who you are, you want to stand out. Adobe believes that creativity and technology empower everyone to reach their full potential. Whether you are an enterprise marketer, small business owner, creative professional, or just trying to create content for yourself, standing out is the key driver behind feeling successful and accomplished. VALUES Create the Future Creativity is in Adobe's DNA. We constantly look around the corner to see what is possible. We are builders, makers, and inventors driven by deep empathy for our customers. We have the courage to disrupt

In [103]:
### Chunking###
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(cleaned_texts)

In [104]:
chunks

[["ADOBE Brand Voice & Tone Guidelines — For RAG Integration MISSION Changing the world through personalized digital experiences. Adobe empowers everyone — from individual creators to global enterprises — to bring their ideas to life through creativity and technology. Nurturing creativity is at the heart of everything Adobe does, for employees as well as the individuals, businesses, and communities it serves. BRAND BELIEF No matter who you are, you want to stand out. Adobe believes that creativity and technology empower everyone to reach their full potential. Whether you are an enterprise marketer, small business owner, creative professional, or just trying to create content for yourself, standing out is the key driver behind feeling successful and accomplished. VALUES Create the Future Creativity is in Adobe's DNA. We constantly look around the corner to see what is possible. We are builders, makers, and inventors driven by deep empathy for our customers. We have the courage to disrup

In [105]:
!pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Flatten the chunks list into a single list of strings
flat_chunks = [item for sublist in chunks for item in sublist]

doc_embeddings = embedder.encode(flat_chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [106]:
import faiss

### Storing the content in FAISS###
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

In [107]:
!pip install faiss-cpu

In [108]:
### Querying the Guidelines###
query = "What are Adobe's tone variations?"
query_embedding = embedder.encode([query])
D, I = index.search(query_embedding, k=1)  # retrieve top 3 chunks
retrieved_docs = [flat_chunks[i] for i in I[0]]

In [109]:
print(retrieved_docs)

["ADOBE Brand Voice & Tone Guidelines — For RAG Integration MISSION Changing the world through personalized digital experiences. Adobe empowers everyone — from individual creators to global enterprises — to bring their ideas to life through creativity and technology. Nurturing creativity is at the heart of everything Adobe does, for employees as well as the individuals, businesses, and communities it serves. BRAND BELIEF No matter who you are, you want to stand out. Adobe believes that creativity and technology empower everyone to reach their full potential. Whether you are an enterprise marketer, small business owner, creative professional, or just trying to create content for yourself, standing out is the key driver behind feeling successful and accomplished. VALUES Create the Future Creativity is in Adobe's DNA. We constantly look around the corner to see what is possible. We are builders, makers, and inventors driven by deep empathy for our customers. We have the courage to disrupt

In [110]:
import requests

In [111]:
API_URL = "https://api-inference.huggingface.co/models/tiiuae/falcon-7b-instruct"

In [118]:
headers = {"Authorization": "yourAPIKEY"}

In [113]:
adobe_guidelines = """
Adobe Brand Voice Guidelines:
- Conversational, human, inspiring, progressive, creative
- Avoid jargon, arrogance, corporate speak
- Tone variations: confident, encouraging, enthusiastic, aspirational, playful
- AI messaging: center human creativity, AI as productivity enhancer
"""

In [114]:
# Step 2: Generate candidate response (simulate RAG pipeline)
def generate_candidate_response(user_prompt, retrieved_context):
    candidate_response = f"Based on {retrieved_context}, here’s a draft for: {user_prompt}"
    return candidate_response

In [115]:
def judge_response(user_prompt, retrieved_context, candidate_response):
    judge_prompt = f"""
You are an evaluator judging whether a response aligns with Adobe’s Brand Voice Guidelines.

Original Prompt:
{user_prompt}

Retrieved Context:
{retrieved_context}

Candidate Response:
{candidate_response}

Evaluate on:
- Faithfulness to Adobe guidelines (1–5)
- Tone compliance (1–5)
- Clarity and readability (1–5)
- Creativity and originality (1–5)

Provide reasoning for each score, then give an overall judgment in JSON format:
{{
  "faithfulness": <score>,
  "tone": <score>,
  "clarity": <score>,
  "creativity": <score>,
  "comments": "<your reasoning>"
}}
"""
    payload = {"inputs": judge_prompt}
    response = requests.post(API_URL, headers=headers, json=payload)

    if not response.ok:
        print(f"API request failed with status code {response.status_code}: {response.text}")
        return {}

    return response.json()

In [116]:
### Step 4: Example usage:
user_prompt = "How do I apply Adobe brand colors in a presentation?"
retrieved_context ="Adobe guidelines on color usage: Use Adobe Red sparingly as an accent color."
candidate_response = generate_candidate_response(user_prompt, retrieved_context)
judgment = judge_response(user_prompt, retrieved_context, candidate_response)

API request failed with status code 404: <!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</title>
</head>
<body>
<pre>Cannot POST /models/tiiuae/falcon-7b-instruct</pre>
</body>
</html>



In [117]:
print("Candidate Response:\n", candidate_response)
print("\nLLM Judge Output:\n", judgment)

Candidate Response:
 Based on Adobe guidelines on color usage: Use Adobe Red sparingly as an accent color., here’s a draft for: How do I apply Adobe brand colors in a presentation?

LLM Judge Output:
 {}
